# Sector showcase: suitability and site selection on the server

All analysis in this notebook runs on the course server. Your laptop only shows the results.

1. Pick a sector template (water, agriculture, roads, education, health, nature-based solutions).
2. Draw or type your area of interest, anywhere in the world.
3. Look at the criteria and weights, change them, and run.
4. See the suitability map and the best candidate sites, then compare with different weights.

In [ ]:
import json, os, requests, leafmap
import geopandas as gpd

API = os.environ.get("PROCESSES_URL", "http://pygeoapi:80")   # inside the server network
PRESETS = "/home/jovyan/presets"                               # read-only copy of the templates

def run(process, inputs):
    r = requests.post(f"{API}/processes/{process}/execution", json={"inputs": inputs}, timeout=900)
    if not r.ok:
        raise RuntimeError(r.json().get("description", r.text))
    return r.json()

presets = {p[:-5]: json.load(open(f"{PRESETS}/{p}")) for p in sorted(os.listdir(PRESETS))}
for pid, p in presets.items():
    print(f"{pid:30s} {p['sector']:25s} {p['question']}")

## 1. Choose a template and an area
Change `PRESET` and `AOI` (bbox: min lon, min lat, max lon, max lat). Keep the area under about 50 km x 50 km for class demos.

In [ ]:
PRESET = "education-school-siting"
AOI = [39.10, 21.40, 39.30, 21.60]   # example: part of Jeddah. Use any area.

preset = presets[PRESET]
inputs = json.loads(json.dumps(preset["inputs"]))
print(preset["title"], "-", preset["question"])
for c in inputs.get("criteria", []):
    print(f"  weight {c['weight']:>4}  {c['name']}")

## 2. Optional: set weights with AHP
Say how much more important each criterion is than the others (1 equal, 3 moderately, 5 strongly, 9 extremely). The consistency ratio must be below 0.10.

In [ ]:
import sys; sys.path.insert(0, "/home/jovyan/processes-lib")
from processes.mcda import ahp_weights

names = [c["name"] for c in inputs.get("criteria", [])]
# Example for the 4 school criteria: distance to school vs population vs road vs slope.
matrix = [[1,   1,   3,   5],
          [1,   1,   3,   5],
          [1/3, 1/3, 1,   3],
          [1/5, 1/5, 1/3, 1]]
if len(matrix) == len(names):
    w, cr = ahp_weights(matrix)
    for n, x in zip(names, w): print(f"{x:.3f}  {n}")
    print("consistency ratio", round(cr, 3), "OK" if cr < 0.10 else "too inconsistent, revise")
    inputs["ahp"] = matrix

## 3. Run on the server

In [ ]:
key = "features" if preset["process"] == "buffer-screen" else "aoi"
inputs[key] = AOI
result = run(preset["process"], inputs)
{k: v for k, v in result.items() if k not in ("sites", "buffer")}

## 4. Map the result

In [ ]:
TILES = os.environ.get("TILES_URL", "https://" + os.environ.get("DOMAIN", "localhost") + "/tiles")

def local(url):
    """Outputs live on the server disk; TiTiler reads them directly."""
    return "/outputs/" + url.rsplit("/", 1)[-1]

m = leafmap.Map(center=[(AOI[1]+AOI[3])/2, (AOI[0]+AOI[2])/2], zoom=11)
if "suitability_raster" in result:
    m.add_cog_layer(local(result["suitability_raster"]), name="Suitability", colormap_name="rdylgn",
                    titiler_endpoint=TILES, rescale="0,1")
if result.get("sites", {}).get("features"):
    m.add_geojson(result["sites"], layer_name="Candidate sites")
    display(gpd.GeoDataFrame.from_features(result["sites"]["features"])[["rank", "area_ha", "mean_score"]])
if "distance_raster" in result:
    m.add_cog_layer(local(result["distance_raster"]), name="Distance to facility (m)", colormap_name="magma",
                    titiler_endpoint=TILES)
m

## 5. Sensitivity: does the answer change if priorities change?
Re-run with one weight doubled and compare the top sites. Robust sites appear in both lists.

In [ ]:
if preset["process"] == "suitability":
    alt = json.loads(json.dumps(inputs)); alt.pop("ahp", None)
    alt["criteria"][0]["weight"] *= 2
    r2 = run("suitability", alt)
    a = gpd.GeoDataFrame.from_features(result["sites"]["features"], crs=4326)
    b = gpd.GeoDataFrame.from_features(r2["sites"]["features"], crs=4326)
    stable = a[a.intersects(b.union_all())] if len(a) and len(b) else a.iloc[0:0]
    print(f"{len(stable)} of {len(a)} sites remain in the top list when '{alt['criteria'][0]['name']}' weight doubles")

## 6. Follow-up analysis
Some templates have a second step, e.g. after school siting, measure how many people live beyond 5 km of a school.

In [ ]:
if "follow_up" in preset:
    fu = preset["follow_up"]
    fu_inputs = {**fu["inputs"], "aoi": AOI}
    r3 = run(fu["process"], fu_inputs)
    print(json.dumps({k: v for k, v in r3.items() if k != "sites"}, indent=2))